# Lekcija 5: Momenti slike

Kada možemo izdvojiti zasebno područje (Lekcija 4), momenti nam omogućuju opis njegova oblika pomoću nekoliko brojeva: površine, težišta, orijentacije, pa čak i opisa koji ostaje isti pri translaciji, skaliranju i rotaciji. Momenti su klasična, računalno nezahtjevna alternativa značajkama dobivenima učenjem (koje ćemo obraditi kasnije) za jednostavnu usporedbu oblika.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Što je moment?

Za binarnu sliku sirovi moment $m_{pq}$ definira se kao

$$m_{pq} = \sum_{x,y} x^p y^q \, I(x, y)$$

gdje je $I(x,y)$ jednak 1 unutar oblika, a 0 drugdje. Nekoliko posebnih slučajeva predstavlja već poznate veličine:

- $m_{00}$ = površina (broj piksela)
- ($\bar{x}, \bar{y}) = (m_{10}/m_{00}, m_{01}/m_{00})$ daje težište

**Red** momenta je $p+q$. Moment nultog reda je $m_{00}$; momenti prvog reda su $m_{10}$ i $m_{01}$; a momenti drugog reda su $m_{20}$, $m_{02}$ i $m_{11}$. Funkcija `cv2.moments` izračunava sve njih u jednom pozivu, zajedno sa *središnjim* momentima $\mu_{ij}$, koji su invarijantni na translaciju, te *Huovim momentima*, koji su invarijantni na translaciju, skaliranje i rotaciju.


## Ručni izračun sirovih momenata

Prethodna formula samo je ponderirani zbroj po koordinatama piksela. Za binarnu sliku to znači: pronaći koordinate $(x,y)$ svakog piksela objekta, a zatim zbrojiti $x^py^q$ za sve te piksele. Prikazat ćemo to na maloj slici veličine 3 × 3 s 3 piksela objekta, izravnim izračunom pomoću biblioteke NumPy i provjerom rezultata funkcijom `cv2.moments`.


In [ ]:
im_tiny = np.array([
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 0],
], dtype=np.uint8)

ys, xs = np.nonzero(im_tiny)       # (row, col) = (y, x) of every foreground pixel

m00 = len(xs)                 # area: just a count of foreground pixels
m10 = xs.sum()                # sum of x^1 y^0 over foreground pixels
m01 = ys.sum()                # sum of x^0 y^1 over foreground pixels

m = cv2.moments(im_tiny, binaryImage=True)

print(f'foreground pixels (x, y): {[(int(x), int(y)) for x, y in zip(xs, ys)]}')
print(f'moments:     m00 = {m00}   m10 = {m10}   m01 = {m01}')
print(f"cv2.moments: m00 = {m['m00']:.0f}   m10 = {m['m10']:.0f}   m01 = {m['m01']:.0f}  <-- same as previous line")
print(f'centroid = ({m10 / m00:.2f}, {m01 / m00:.2f})')

## Izduženo, rotirano područje

Nacrtajmo rotiranu elipsu kako bismo njezinu orijentaciju mogli lako procijeniti pogledom i usporediti s rezultatom izračunatim pomoću momenata.


In [ ]:
im_binary = np.zeros((200, 200), dtype=np.uint8)
pixval = 1  # or 255, but note that it will make the moments bigger 
center = (100, 100)
axes_len = (70, 25)
angle_deg = 30
cv2.ellipse(im_binary, center, axes_len, angle_deg, 0, 360, pixval, cv2.FILLED)

plt.imshow(255 * im_binary, cmap='gray')
plt.title(f'Ellipse drawn at {angle_deg} degrees')
plt.axis('off')
plt.show()

Funkcija `cv2.ellipse` iz biblioteke OpenCV stvara neobične efekte uz rub. Ako ih želite ukloniti, možete jednostavno sami napisati kod za crtanje elipse.


In [ ]:
def draw_ellipse(im, cen, axes, angle_deg, val):
    h, w = im.shape
    Y, X = np.mgrid[0:h, 0:w]
    ang = np.deg2rad(angle_deg)
    XX = np.cos(ang) * (X-cen[0]) + np.sin(ang) * (Y-cen[1])
    YY = -np.sin(ang) * (X-cen[0]) + np.cos(ang) * (Y-cen[1])
    return np.uint8(val * (np.sqrt( (XX**2) + ((axes[0] / axes[1])**2)*(YY**2)) < (1.006)*axes[0]))

im_binary2 = np.zeros((200, 200), dtype=np.uint8)
im_binary2 = draw_ellipse(im_binary2, center, axes_len, angle_deg, pixval)
im_binary = im_binary2  # Let's replace the ellipse with our fixed ellipse

plt.imshow(255 * im_binary2, cmap='gray')
ccen = (165,90) # location of red circle when ellipse angle is 0 degrees
ccen_rotx = center[0] + np.cos(np.deg2rad(angle_deg)) * (ccen[0]-center[0]) - np.sin(np.deg2rad(angle_deg)) * (ccen[1]-center[1])
ccen_roty = center[1] + np.sin(np.deg2rad(angle_deg)) * (ccen[0]-center[0]) + np.cos(np.deg2rad(angle_deg)) * (ccen[1]-center[1])
circle = plt.Circle((ccen_rotx, ccen_roty), 10, color='red', fill=False, linewidth=1, alpha=1.0)
plt.gca().add_patch(circle)
plt.title(f'Fixed ellipse - Look closely inside red circle')
plt.axis('off')
plt.show()

## Izračun sirovih momenata

Sirovi momenti lako se izračunavaju: dovoljno je zbrajati po pikselima slike, kao što je prethodno objašnjeno.


In [ ]:
def compute_moments(im):
    summ = 0
    sumx = 0
    sumy = 0    
    for y in range(im.shape[0]):
        for x in range(im.shape[1]):
            v = im[y,x]
            if v > 0:
                summ += 1
                sumx += x
                sumy += y
    out = {}
    out['m00'] = summ
    out['m10'] = sumx
    out['m01'] = sumy
    return out

mm = compute_moments(im_binary)
m = cv2.moments(im_binary, binaryImage=True)
print(f"moments:     m00 = {mm['m00']:.0f}  m10 = {mm['m10']:.0f}  m01 = {mm['m01']:.0f}")
print(f"cv2.moments: m00 = {m['m00']:.0f}  m10 = {m['m10']:.0f}  m01 = {m['m01']:.0f}  <-- same as previous line")

## Površina i težište iz momenata


In [ ]:
m = cv2.moments(im_binary, binaryImage=True)

area = m['m00']
cx = m['m10'] / m['m00']
cy = m['m01'] / m['m00']

print(f'area (m00)   = {area:.0f} pixels')
print(f'centroid     = ({cx:.1f}, {cy:.1f})')

plt.imshow(im_binary, cmap='gray')
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Centroid from moments')
plt.axis('off')
plt.show()

## Središnji momenti

Sirovi momenti $m_{pq}$ ovise o položaju oblika u slici &mdash; pomicanjem oblika mijenja se svaki $m_{ij}$ osim $m_{00}$. **Središnji momenti** to rješavaju tako da mjere u odnosu na težište samog oblika $(\bar x, \bar y)$, umjesto u odnosu na ishodište slike:

$$\mu_{pq} = \sum_{x,y} (x-\bar x)^p\,(y-\bar y)^q\, I(x,y)$$

Ponovni izračun ovog zbroja od početka zahtijevao bi ponovni obilazak svakog piksela. Umjesto toga, identitet za pomak ishodišta &mdash; analogon izraza $\mathrm{Var}(X) = E[X^2] - E[X]^2$ za momente slike &mdash; omogućuje izračun središnjih momenata izravno iz sirovih momenata:

$$\mu_{20} = M_{20} - \bar x\,m_{10}, \qquad \mu_{02} = m_{02} - \bar y\,m_{01}, \qquad \mu_{11} = m_{11} - \bar x\,m_{01}$$

Funkcija `cv2.moments` vraća i sirove (`m['m20']`, ...) i središnje (`m['mu20']`, ...) momente u istom rječniku, pa ih u praksi ne morate računati ručno.


In [ ]:
mu20_manual = m['m20'] - cx * m['m10']
mu02_manual = m['m02'] - cy * m['m01']
mu11_manual = m['m11'] - cx * m['m01']

print(f'moments:     mu20 = {mu20_manual:.1f}   mu02 = {mu02_manual:.1f}   mu11 = {mu11_manual:.1f}')
print(f"cv2.moments: mu20 = {m['mu20']:.1f}   mu02 = {m['mu02']:.1f}   mu11 = {m['mu11']:.1f}")

## Orijentacija iz središnjih momenata

Središnji momenti $\mu_{20}$, $\mu_{02}$, $\mu_{11}$ opisuju raspršenost oblika oko njegova težišta — u osnovi njegovu matricu kovarijance, koju ćemo ponovno razmotriti u Lekciji 6. Kut glavne osi (smjera najveće raspršenosti) jest

$$\theta = \frac{1}{2}\,\mathrm{atan2}\!\left(2\mu_{11},\; \mu_{20} - \mu_{02}\right)$$


In [ ]:
theta = 0.5 * np.arctan2(2 * m['mu11'], m['mu20'] - m['mu02'])
theta_deg = np.degrees(theta)

print(f'orientation from moments = {theta_deg:.1f} degrees')
print(f'angle used to draw the ellipse = {angle_deg} degrees')

# The eigenvalues of the (normalized) covariance matrix of central moments
# give the axis lengths of the equivalent ellipse: axis = 4*sqrt(eigenvalue).
cov = np.array([[m['mu20'], m['mu11']], [m['mu11'], m['mu02']]]) / m['m00']
eigvals, _ = np.linalg.eigh(cov)
semi_major = 2 * np.sqrt(eigvals[-1])

theta = np.deg2rad(angle_deg)  ####################
dx, dy = semi_major * np.cos(theta), semi_major * np.sin(theta)

print(semi_major, cx, cy, dx, dy)  ######################
plt.imshow(im_binary, cmap='gray')
plt.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c='red', linewidth=2)
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Major axis recovered from moments')
plt.axis('off')
plt.axis('equal')
plt.show()

## Huovi momenti: opis oblika invarijantan na položaj, veličinu i orijentaciju

**Huovi momenti** (`cv2.HuMoments`) skup su od 7 vrijednosti izračunatih iz središnjih momenata koje ostaju gotovo iste bez obzira na položaj, veličinu i rotaciju oblika. Zbog toga su korisni za usporedbu dvaju oblika bez prethodnog poravnanja.

Kako bismo to vidjeli, izradit ćemo tri inačice oblika psa: izvornu, translatiranu te rotiranu i skaliranu. Usporedit ćemo njihove Huove momente međusobno te s momentima drukčijeg oblika — mačke.


In [ ]:
def warp(im, mat):
    rows, cols = im.shape[:2]
    return cv2.warpAffine(im, np.float32(mat), (cols, rows))

im_original = cv2.imread('../img/dog_clipart.png', cv2.IMREAD_GRAYSCALE)
im_translated = warp(im_original, [[1, 0, 10], [0, 1, 20]])
im_rotated_scaled = warp(im_original, cv2.getRotationMatrix2D([60, 60], angle=45, scale=0.5))
im_different_shape = cv2.imread('../img/cat_clipart.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [im_original, im_translated, im_rotated_scaled, im_different_shape],
    ['Original', 'Translated', 'Rotated + scaled', 'Different shape'],
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: <a href="https://www.publicdomainpictures.net" target="_blank" rel="noopener">Public domain pictures</a></p>


Sada izračunajmo Huove momente.


In [ ]:
def hu_log(im):
    m = cv2.moments(im, binaryImage=True)
    hu = cv2.HuMoments(m).flatten()
    # log-scale since raw Hu moments span many orders of magnitude
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

# Format with a fixed width (":8.3f") to align the values across rows.
label_width, col_width, n_hu = 18, 8, 7
print(' ' * (label_width + 2) + 'Hu moments'.center(col_width * n_hu))
print(f'{"image":>{label_width}}  ' + ''.join(f'{i:>{col_width}}' for i in range(1, n_hu + 1)))
for im, name in [
    (im_original, 'original'),
    (im_translated, 'translated'),
    (im_rotated_scaled, 'rotated+scaled'),
    (im_different_shape, 'different shape'),
]:
    row = ''.join(f'{v:{col_width}.3f}' for v in hu_log(im))
    print(f'{name:>{label_width}}  {row}')

Primijetite da translatirana te rotirana i skalirana inačica daju slične Huove momente, dok drukčiji oblik daje znatno drukčije vrijednosti. Upravo to svojstvo invarijantnosti čini Huove momente korisnima za usporedbu oblika.


### Zadaci

1. Upotrijebite `cv2.findContours` kako biste dobili konturu jednog područja iz binarne slike iz Lekcije 4, a zatim pozovite `cv2.moments` nad *konturom* umjesto nad cijelom binarnom maskom. Usporedite dobiveno težište s ovdje izračunatim.
2. Nacrtajte oblik koji je zrcaljen umjesto rotiran. Jesu li njegovi Huovi momenti i dalje bliski izvornima? (Pomoć: razmislite o tome koje informacije o determinanti i paritetu momenti obuhvaćaju, a koje ne.)
